# M5 GuideLLM independent serving cross-check

Status: **PREPARED_FOR_KAGGLE**. Run only after the corresponding M4 Qwen principal cell is accepted. Use a fresh T4 x2 session, Internet enabled, and private `HF_TOKEN` plus `GUIDELLM_SHARD_ID` secrets. GuideLLM runs from reviewed commit `fc2dbe9edd4f7f1a4e9ccd752f6f43591adbcb73` in a separate client virtual environment; it is not installed into the canonical server runtime.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, venv
from pathlib import Path
from kaggle_secrets import UserSecretsClient

EXPECTED_SOURCE_COMMIT = 'SOURCE_COMMIT_TO_PIN_AFTER_REVIEW'; GUIDELLM_COMMIT = 'fc2dbe9edd4f7f1a4e9ccd752f6f43591adbcb73'
WORK = Path('/kaggle/working'); SOURCE = WORK/'kaggle-vllm-source'; GUIDE = WORK/'guidellm-source'; GUIDE_VENV = WORK/'guidellm-client-venv'; RUNTIME = WORK/'kaggle-vllm-runtime'; CACHE = WORK/'kaggle-vllm-cache'; HF_HOME = WORK/'hf-cache'
assert Path('/kaggle').is_dir() and all(not path.exists() for path in (SOURCE, GUIDE, GUIDE_VENV, RUNTIME)), 'Use a fresh Kaggle session'
secrets = UserSecretsClient(); token = secrets.get_secret('HF_TOKEN'); shard_id = secrets.get_secret('GUIDELLM_SHARD_ID'); assert token and shard_id
os.environ['HF_TOKEN'] = token; os.environ['HF_HOME'] = str(HF_HOME); HF_HOME.mkdir()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'kaggle-vllm[hub]==0.2.0'], check=True)
for path, url, commit in ((SOURCE, 'https://github.com/kaggle-vllm/kaggle-vllm.git', EXPECTED_SOURCE_COMMIT), (GUIDE, 'https://github.com/vllm-project/guidellm.git', GUIDELLM_COMMIT)):
    subprocess.run(['git', 'init', str(path)], check=True); subprocess.run(['git', '-C', str(path), 'remote', 'add', 'origin', url], check=True); subprocess.run(['git', '-C', str(path), 'fetch', '--depth', '1', 'origin', commit], check=True); subprocess.run(['git', '-C', str(path), 'checkout', '--detach', 'FETCH_HEAD'], check=True); assert subprocess.check_output(['git', '-C', str(path), 'rev-parse', 'HEAD'], text=True).strip() == commit; assert not subprocess.check_output(['git', '-C', str(path), 'status', '--porcelain'], text=True).strip()
plan = json.loads((SOURCE/'research/M4_EXECUTION_PLAN.json').read_text()); matches = [item for item in plan['guidellm_crosscheck_order'] if item['shard_id'] == shard_id]; assert len(matches) == 1; shard = matches[0]
RUNTIME.mkdir(); CACHE.mkdir(); manifest = RUNTIME/'runtime.json'; boot = ['kaggle-vllm', 'bootstrap', '--strict', '--staged', str(RUNTIME/'staged'), '--overlay', str(RUNTIME/'overlay'), '--cache', str(CACHE), '--manifest', str(manifest)]
subprocess.run(boot + ['--dry-run'], check=True); subprocess.run(boot, check=True); runtime = json.loads(manifest.read_text()); RUN_ENV = dict(os.environ); RUN_ENV.update(runtime['runtime_environment']); RUN_ENV['PYTHONPATH'] = str(SOURCE/'src') + os.pathsep + str(SOURCE) + os.pathsep + RUN_ENV.get('PYTHONPATH', '')
venv.EnvBuilder(with_pip=True, system_site_packages=True).create(GUIDE_VENV); guide_python = GUIDE_VENV/'bin/python'; subprocess.run([str(guide_python), '-m', 'pip', 'install', '--no-cache-dir', str(GUIDE)], check=True); subprocess.run([str(guide_python), '-m', 'pip', 'check'], check=True); guide_exe = GUIDE_VENV/'bin/guidellm'; subprocess.run([str(guide_exe), '--help'], check=True)
print(json.dumps({'source_commit': EXPECTED_SOURCE_COMMIT, 'guidellm_commit': GUIDELLM_COMMIT, 'shard': shard}, indent=2))

In [ ]:
output = WORK/'guidellm-evidence'/shard_id
command = [sys.executable, str(SOURCE/'scripts/kaggle_guidellm_crosscheck.py'), '--repository', str(SOURCE), '--output-dir', str(output), '--model-key', shard['model_key'], '--workload', shard['workload'], '--concurrency', str(shard['concurrency']), '--tensor-parallel-size', str(shard['tensor_parallel_size']), '--repetition', str(shard['repetition']), '--source-identity', EXPECTED_SOURCE_COMMIT, '--guidellm-executable', str(guide_exe), '--guidellm-source', str(GUIDE)]
completed = subprocess.run(command, cwd=SOURCE, env=RUN_ENV, check=False); assert output.is_dir(); subprocess.run([sys.executable, '-m', 'kaggle_vllm.research', 'verify-hashes', str(output)], cwd=SOURCE, env=RUN_ENV, check=True)
archive = Path(shutil.make_archive(str(output), 'zip', output)); print(json.dumps({'runner_returncode': completed.returncode, 'zip': str(archive), 'zip_sha256': hashlib.sha256(archive.read_bytes()).hexdigest()}, indent=2)); assert completed.returncode in {0, 2}

Download the ZIP and executed notebook. Preserve GuideLLM's TTFT/ITL/TPOT and throughput definitions as GuideLLM definitions; comparison with the primary client requires explicit reconciliation and accepted repeated evidence.